# Delta Robot Kinematics, Configuration Space, Workspace and Singularity Analysis

## Degrees of Freedom

The delta robot has 3 degrees of freedom. The 3 degrees of freedom are the 3 angles of the 3 arms. The 3 angles are the angles between the arms and the base. The 3 angles are denoted by $\theta_1$, $\theta_2$, and $\theta_3$. The 3 angles are the joint angles of the delta robot. The 3 angles are the configuration space of the delta robot. The configuration space of the delta robot is the space of all possible joint angles of the delta robot. The configuration space of the delta robot is a 3-dimensional space. The configuration space of the delta robot is a 3-dimensional space because the delta robot has 3 degrees of freedom.

<div style="text-align: center;">
  <img src="images/delta_robot_modern_robotics.png" alt="Delta Robot">
</div>

In [ ]:
# Import block
!pip install nbformat numpy plotly
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

## Robot Constants

In [ ]:
# Robot Constants
Base_Inradius = 100  # Base Triangle Circumradius [mm]
End_Effector_Inradius = 32.5  # End Effector Triangle Circumradius [mm]
SB = 2*np.sqrt(3)*Base_Inradius  # Base Equilateral Triangle Side Length [mm]
SP = 2*np.sqrt(3)*End_Effector_Inradius  # Platform Equilateral Triangle Side Length [mm]
L = 100.0  # Active Link Length [mm]
ELL = 200.0  # Passive Link Length [mm]
H = 42.84670  # Passive Link Width [mm]
WB = (np.sqrt(3) / 6) * SB
UB = (np.sqrt(3) / 3) * SB
WP = (np.sqrt(3) / 6) * SP
UP = (np.sqrt(3) / 3) * SP

print(f"SB: {SB} mm (Base Equilateral Triangle Side Length)")
print(f"SP: {SP} mm (Platform Equilateral Triangle Side Length)")
print(f"L: {L} mm (Active Link Length)")
print(f"ELL: {ELL} mm (Passive Link Length)")
print(f"H: {H} mm (Passive Link Width)")
print(f"WB: {WB} mm (Base Triangle Height)")
print(f"UB: {UB} mm (Base Triangle Circumradius)")
print(f"WP: {WP} mm (Platform Triangle Height)")
print(f"UP: {UP} mm (Platform Triangle Circumradius)")

# Inverse Kinematics

- https://people.ohio.edu/williams/html/PDF/DeltaKin.pdf
- https://hypertriangle.com/~alex/delta-robot-tutorial/ (C Implementation)

In [ ]:
def IK(x,y,z):
  tan30 = 1 / np.sqrt(3) #np.tan(np.pi / 6)
  cos120 = -0.5 #np.cos(2 * np.pi / 3) # cos(120 degrees)
  sin120 = np.sqrt(3) / 2 #np.sin(2 * np.pi / 3) # sin(120 degrees)
  theta1 = 0
  theta2 = 0
  theta3 = 0
  def calc_angleYZ(x0,y0,z0):
    y1 = -0.5 * tan30 * SB
    y0 -= 0.5 * tan30 * SP
    a = (x0**2 + y0**2 + z0**2 + L**2 - ELL**2 - y1**2) / (2 * z0)
    b = (y1 - y0) / z0
    d = -(a + b * y1) * (a + b * y1) + L * (b * b * L + L)
    if (d < 0):
      return None
    yj = (y1 - a * b - np.sqrt(d)) / (b * b + 1)
    zj = a + b * yj
    theta = np.arctan(-zj / (y1 - yj)) + (np.pi if yj > y1 else 0.0)
    return theta
  # Inverse Kinematics
  theta1 = calc_angleYZ(x, y, z)
  if (theta1 != None):
    theta2 = calc_angleYZ(
      x0=x*cos120 + y*sin120,
      y0=y*cos120 - x*sin120,
      z0=z
    ) # Rotate coords by +120 deg
  else:
    print(f'Theta1 has no solution for input ({x},{y},{z})')
    return None
  if (theta2 != None):
    theta3 = calc_angleYZ(
      x0=x*cos120 - y*sin120,
      y0=y*cos120 + x*sin120,
      z0=z
    )
  else:
    print(f'Theta2 has no solution for input ({x},{y},{z})')
    return None
  if (theta3 == None):
    print(f'Theta3 has no solution for input ({x},{y},{z})')
    return None
  return np.array([theta1, theta2, theta3])

def IK_alternative(x,y,z):
  # The analytical solution for IK uses the constraint equations found from the vector-loop closure equations
  # E_i * cos(theta_i) + F_i *  sin(theta_i) + G_i = 0
  # Variables obtained from the Vector-Loop Closure Equations
  a = WB - UP
  b = (SP / 2) - ((np.sqrt(3) / 2) * WP)
  c = WP - (WB / 2)
  E = (
    2*L*(y+a), # E1
    -L*(np.sqrt(3)*(x+b)+y+c),  # E2
    L*(np.sqrt(3)*(x-b)-y-c)  # E3
    )
  F = (
    2*z*L, # F1
    2*z*L,  # F2
    2*z*L,  # F3
  )
  G = (
      x**2 + y**2 + z**2 + a**2 + L**2 + 2*y*a - ELL**2,  # G1
      x**2 + y**2 + z**2 + b**2 + c**2 + L**2 + 2*(x*b+y*c) - ELL**2,  # G2
      x**2 + y**2 + z**2 + b**2 + c**2 + L**2 + 2*(-x*b+y*c) - ELL**2,  # G3
  )
  thetas = []
  # Tangent Half-Angle Substitution
  for i in range(3):
    D = E[i]**2 + F[i]**2 - G[i]**2
    if D < 0:
      print(f'No solution for input ({x},{y},{z})')
      thetas.append(0)
      continue
    t_plus = (-F[i] + np.sqrt(D)) / (G[i] - E[i])
    t_minus = (-F[i] - np.sqrt(D)) / (G[i] - E[i])
    theta_plus = 2 * np.arctan(t_plus)
    theta_minus = 2 * np.arctan(t_minus)
    # Pick the solution with the knees "kinked" out, the angle should be closer to 0
    thetas.append(theta_plus if abs(theta_plus) < abs(theta_minus) else theta_minus)
  # return a tuple of the 3 joint angles
  return np.array([thetas[0], thetas[1], thetas[2]])

up_position = IK(0, 0, -100)
print(f'IK(0,0,-100): {np.round(up_position,3)}')
down_position = IK(0, 0, -200)
print(f'IK(0,0,-200): {np.round(down_position,3)}')
# Test the IK function with a list of points
# Straight-up and down between Z=-200 and Z= -100
straight_up_down = [(0, 0, z) for z in np.linspace(-100, -200, 50)]
theta_values = np.array([IK(*point) for point in straight_up_down])
print(f'Trajectory from Z=-100 to Z=-200: {np.round(theta_values,3)} [rad]')
assert np.allclose(theta_values[0], up_position, atol=1e-6), f'Expected first point to be up_position: {theta_values[0]} != {up_position}'
assert np.allclose(theta_values[-1], down_position, atol=1e-6), f'Expected last point to be down_position: {theta_values[-1]} != {down_position}'

# Forward Kinematics


In [ ]:
def FK_alt(th1,th2,th3):
  def virtual_sphere_centers(th1, th2, th3):
    # thetas are the joint angles [rad]
    return (
      (0, -WB-L*np.cos(th1)+UP, -L*np.sin(th1)), # Sphere 1
      ((np.sqrt(3)/2)*(WB+L*np.cos(th2))-(SP/2), (1/2) * (WB+L*np.cos(th2))-WP, -L*np.sin(th2)),  # Sphere 2
      (-(np.sqrt(3)/2)*(WB+L*np.cos(th3))+(SP/2), (1/2) * (WB+L*np.cos(th3))-WP, -L*np.sin(th3)),  # Sphere 3
    )
  # First determine the centers of the 3 virtual spheres
  sphere_centers = virtual_sphere_centers(th1, th2, th3)
  x1, y1, z1 = sphere_centers[0]
  x2, y2, z2 = sphere_centers[1]
  x3, y3, z3 = sphere_centers[2]

  # Determine if the all Z heights of the 3 spheres are the same to determine which algorithm to use
  same_z_height = all([center[2] == sphere_centers[0][2] for center in sphere_centers])

  if same_z_height:
    # Simplified 3 Sphere Intersection Algorithm
    assert z1 == z2 == z3
    z_n = z1 # All Z heights are the same
    a = 2*(x3 - x1)
    b = 2*(y3 - y1)
    c = ELL**2 - ELL**2 - x1**2 - y1**2 + x3**2 + y3**2
    d = 2*(x3 - x2)
    e = 2*(y3 - y2)
    f = ELL**2 - ELL**2 - x2**2 - y2**2 + x3**2 + y3**2
    denominator = (a*e - b*d)
    if denominator == 0:
      raise ValueError('FK(S): Denominator is zero, the spheres do not intersect')
    x = (c*e - b*f) / denominator
    y = (a*f - c*d) / denominator
    B = -2 * z_n
    C = z_n**2 - ELL**2 + (x - x1)**2 + (y - y1)**2
    discriminant = B**2 - 4*C
    if discriminant < 0:
      raise ValueError('FK(S): No real solutions, the spheres do not intersect')
    z_plus = (-B + np.sqrt(discriminant)) / 2
    z_minus = (-B - np.sqrt(discriminant)) / 2
    # Choose the Z value that is in the correct workspace (-Z)
    z = z_plus if z_plus < 0 else z_minus
    return np.array([x,y,z])
  else:
    # 3 Sphere Intersection Algorithm
    a11 = 2*(x3-x1)
    a12 = 2*(y3-y1)
    a13 = 2*(z3-z1)
    a21 = 2*(x3-x2)
    a22 = 2*(y3-y2)
    a23 = 2*(z3-z2)
    b1 = ELL**2 - ELL**2 - x1**2 - y1**2 - z1**2 + x3**2 + y3**2 + z3**2
    b2 = ELL**2 - ELL**2 - x2**2 - y2**2 - z2**2 + x3**2 + y3**2 + z3**2
    # Verify we aren't dividing by zero
    if a13 == 0 or a23 == 0:
      raise ValueError('FK: Division by zero, the spheres do not intersect')
    a1 = (a11/a13) - (a21/a23)
    a2 = (a12/a13) - (a22/a23)
    a3 = (b2/a23) - (b1/a13)
    a4 = -(a2/a1)
    a5 = -(a3/a1)
    a6 = (-a21*a4-a22)/a23
    a7 = (b2-a21*a5)/a23
    a_quad = a4**2 + 1 + a6**2
    b_quad = 2*a4*(a5-x1) - 2*y1 + 2*a6*(a7-z1)
    c_quad = a5*(a5-2*x1) + a7*(a7-2*z1) + x1**2 + y1**2 + z1**2 - ELL**2
    d_quad = b_quad**2 - 4*a_quad*c_quad
    if d_quad < 0:
      raise ValueError('FK: No real solutions, the spheres do not intersect')
    y_plus = (-b_quad+np.sqrt(d_quad))/(2*a_quad)
    x_plus = a4*y_plus + a5
    z_plus = a6*y_plus + a7
    y_minus = (-b_quad-np.sqrt(d_quad))/(2*a_quad)
    x_minus = a4*y_minus + a5
    z_minus = a6*y_minus + a7
    # Choose the Z value that is in the correct workspace (-Z)
    if z_plus < 0:
      z = z_plus
      x = x_plus
      y = y_plus
    else:
      z = z_minus
      x = x_minus
      y = y_minus
    return np.array([x, y, z])

def FK(theta1, theta2, theta3):
  # define some constants
  tan30 = 1 / np.sqrt(3)
  sin30 = 0.5
  tan60 = np.sqrt(3)
  t = (SB - SP) * tan30 / 2

  y1 = -(t + L * np.cos(theta1))
  z1 = -L * np.sin(theta1)

  y2 = (t + L * np.cos(theta2)) * sin30
  x2 = y2 * tan60
  z2 = -L * np.sin(theta2)

  y3 = (t + L * np.cos(theta3)) * sin30
  x3 = -y3 * tan60
  z3 = -L * np.sin(theta3)

  dnm = (y2 - y1) * x3 - (y3 - y1) * x2

  w1 = y1 * y1 + z1 * z1
  w2 = x2 * x2 + y2 * y2 + z2 * z2
  w3 = x3 * x3 + y3 * y3 + z3 * z3

  a1 = (z2 - z1) * (y3 - y1) - (z3 - z1) * (y2 - y1)
  b1 = -((w2 - w1) * (y3 - y1) - (w3 - w1) * (y2 - y1)) / 2

  a2 = -(z2 - z1) * x3 + (z3 - z1) * x2
  b2 = ((w2 - w1) * x3 - (w3 - w1) * x2) / 2

  a = a1 * a1 + a2 * a2 + dnm * dnm
  b = 2 * (a1 * b1 + a2 * (b2 - y1 * dnm) - z1 * dnm * dnm)
  c = (b2 - y1 * dnm) * (b2 - y1 * dnm) + b1 * b1 + dnm * dnm * (z1 * z1 - ELL * ELL)

  d = b * b - 4 * a * c
  if (d < 0):
    # print(
    #     f'FK: No real solution for configuration ({theta1:.2f}/{np.rad2deg(theta1):.1f},{theta2:.2f}/{np.rad2deg(theta2):.1f},{theta3:.2f}/{np.rad2deg(theta3):.1f}) [rad/deg]'
    #     )
    return None

  z = -0.5 * (b + np.sqrt(d)) / a
  x = (a1 * z + b1) / dnm
  y = (a2 * z + b2) / dnm
  return np.array([x,y,z])

test_point = [0.3034318685531616, 0.4903949499130249,0.4903949499130249]
print(f'FK({test_point}): {np.round(FK(*test_point), 3)}')
fd = np.deg2rad(30)
print(f'FK with equal joint angles ({fd:.2f}) [rad]: {np.round(FK(fd,fd,fd), 4)} [mm]')
print(f'FK with different joint angles (-0.25, -0.5, -0.5) [rad]: {np.round(FK(-0.25, -0.5, 0.5),4)} [mm]')

# Plotting the Robot's Configuration

In order to better visualize the robot's configuration, we can plot the robot's links and joints in 3D space.

In [ ]:

def motor_positions():
  return (
    (0,-WB,0), # B1
    ((np.sqrt(3)/2)*WB,WB/2,0), # B2
    (-(np.sqrt(3)/2)*WB, WB/2, 0)  # B3
  )

def base_vertices():
  return (
    (SB/2, -WB, 0), # b1
    (0, UB, 0), # b2
    (-SB/2, -WB, 0) # b3
  )

def platform_vertices(platform_position):
  # platform_position is a tuple (x,y,z)
  x, y, z = platform_position
  # Platform vertices with respect to the given platform position
  return (
    (x, y - UP, z), # P1
    (x + SP/2, y + WP, z), # P2
    (x - SP/2, y + WP, z) # P3
  )

def knee_joints(theta1,theta2,theta3):
  knee_1 = (0, -WB-L*np.cos(theta1), -L*np.sin(theta1))
  knee_2 = ((np.sqrt(3)/2)*(WB+L*np.cos(theta2)), (1/2)*(WB+L*np.cos(theta2)), -L*np.sin(theta2))
  knee_3 = ((-np.sqrt(3)/2)*(WB+L*np.cos(theta3)), (1/2)
            * (WB+L*np.cos(theta3)), -L*np.sin(theta3))
  return (knee_1, knee_2, knee_3)


def get_motor_rectangle_corners(motor_position, angle, motor_width=20, motor_length=40):
    # Local coordinates (centered at 0,0)
    dx = motor_length / 2
    dy = motor_width / 2
    local_corners = np.array([
        [-dx, -dy],
        [dx, -dy],
        [dx,  dy],
        [-dx,  dy]
    ])
    # Create rotation matrix for the given angle
    R = np.array([
        [np.cos(angle), -np.sin(angle)],
        [np.sin(angle),  np.cos(angle)]
    ])
    rotated = (local_corners @ R.T) + np.array(motor_position[:2])
    # Append the constant z coordinate from motor_position
    corners = [(pt[0], pt[1], motor_position[2]) for pt in rotated]
    return corners

def plot_robot(theta1, theta2, theta3, additional_traces=None, save_to_html=False, name=None):
  # Calculate the knee joint positions
  knee_1, knee_2, knee_3 = knee_joints(theta1, theta2, theta3)
  # Calculate the platform position
  try:
    platform_position = FK(theta1, theta2, theta3)
  except Exception as e:
    print(f"Exception for FK({theta1}, {theta2}, {theta3}): {e}")
    return
  # Get the base vertices
  bv1, bv2, bv3 = base_vertices()
  # Get the motor positions
  m1, m2, m3 = motor_positions()
  # Get the platform vertices
  pv1, pv2, pv3 = platform_vertices(platform_position)

  # Create a 3D plot
  fig = go.Figure()

  # --- Base Triangle (Mesh3d) ---
  base_x = [bv1[0], bv2[0], bv3[0]]
  base_y = [bv1[1], bv2[1], bv3[1]]
  base_z = [bv1[2], bv2[2], bv3[2]]
  fig.add_trace(go.Mesh3d(
      x=base_x, y=base_y, z=base_z,
      i=[0], j=[1], k=[2],
      color='lightblue', opacity=0.6,
      name='Base'
  ))

  # --- Platform Triangle (Mesh3d) ---
  plat_x = [pv1[0], pv2[0], pv3[0]]
  plat_y = [pv1[1], pv2[1], pv3[1]]
  plat_z = [pv1[2], pv2[2], pv3[2]]
  fig.add_trace(go.Mesh3d(
      x=plat_x, y=plat_y, z=plat_z,
      i=[0], j=[1], k=[2],
      color='lightgreen', opacity=1.0,
      name='Platform'
  ))

  # Draw a red sphere at the platform position
  fig.add_trace(go.Scatter3d(
        x=[platform_position[0]], y=[platform_position[1]], z=[platform_position[2]],
        mode='markers',
        marker=dict(color='red', size=10),
        name='Platform Center',
        showlegend=False
    ))

  # Add platform vertices as markers with labels
  platform_vertices_list = [pv1, pv2, pv3]
  for idx, pv in enumerate(platform_vertices_list, start=1):
      fig.add_trace(go.Scatter3d(
          x=[pv[0]], y=[pv[1]], z=[pv[2]],
          mode='markers+text',
          marker=dict(color='green', size=H/4),
          text=f'P{idx}',
          textposition='top center',
          name=f'PV {idx}',
          showlegend=False
      ))

  # --- Motor Rectangles ---
  # Determine motor orientations based on the base triangle sides:
  motor_angle1 = np.arctan2(bv1[1] - bv3[1], bv1[0] - bv3[0])
  motor_angle2 = np.arctan2(bv2[1] - bv1[1], bv2[0] - bv1[0])
  motor_angle3 = np.arctan2(bv3[1] - bv2[1], bv3[0] - bv2[0])
  motor_angles = [motor_angle1, motor_angle2, motor_angle3]
  motor_positions_list = [m1, m2, m3]
  motor_colors = ['red', 'green', 'blue']
  motor_ids = ['1', '2', '3']

# For each motor, compute its rectangle and add it as two triangles.
  for m, ang, color, mid in zip(motor_positions_list, motor_angles, motor_colors, motor_ids):
    corners = get_motor_rectangle_corners(m, ang)
    # Split quadrilateral into two triangles:
    # Triangle 1: corners[0], corners[1], corners[2]
    tri1_x = [corners[0][0], corners[1][0], corners[2][0]]
    tri1_y = [corners[0][1], corners[1][1], corners[2][1]]
    tri1_z = [corners[0][2], corners[1][2], corners[2][2]]

    # Triangle 2: corners[0], corners[2], corners[3]
    tri2_x = [corners[0][0], corners[2][0], corners[3][0]]
    tri2_y = [corners[0][1], corners[2][1], corners[3][1]]
    tri2_z = [corners[0][2], corners[2][2], corners[3][2]]

    # Add first triangle
    fig.add_trace(go.Mesh3d(
        x=tri1_x, y=tri1_y, z=tri1_z,
        color=color, opacity=0.7,
        name=f'Motor {mid}',
        showscale=False
    ))
    # Add second triangle
    fig.add_trace(go.Mesh3d(
        x=tri2_x, y=tri2_y, z=tri2_z,
        color=color, opacity=0.7,
        name=f'Motor {mid}',
        showscale=False
    ))
    # Add motor label as text at the motor position
    fig.add_trace(go.Scatter3d(
      x=[m[0]], y=[m[1]], z=[m[2]],
      mode='text',
      text=[mid],
      textposition='middle center',
      textfont=dict(color='black', size=20),
      showlegend=False
    ))
  # --- End of Motor Rectangles ---

  # --- Active Links (lines from motors to knee joints) ---
  knees = [knee_1, knee_2, knee_3]
  for m, k in zip(motor_positions_list, knees):
      fig.add_trace(go.Scatter3d(
          x=[m[0], k[0]],
          y=[m[1], k[1]],
          z=[m[2], k[2]],
          mode='lines',
          line=dict(color='purple', width=H/4),
          name='Active Link' + str(motor_positions_list.index(m) + 1),
          showlegend=False
      ))

      # --- Knee Joints as markers ---
      fig.add_trace(go.Scatter3d(
          x=[k[0]], y=[k[1]], z=[k[2]],
          mode='markers',
          marker=dict(color='black', size=10),
          name='Knee Joint' + str(knees.index(k) + 1),
          showlegend=False
      ))

  # --- Passive Links (rectangles from knee joints to platform vertices) ---
  for k, p in zip(knees, [pv1, pv2, pv3]):
      fig.add_trace(go.Scatter3d(
          x=[k[0], p[0]],
          y=[k[1], p[1]],
          z=[k[2], p[2]],
          mode='lines',
          line=dict(color='orange', width=H),
          name='Passive Link' + str(knees.index(k) + 1),
          showlegend=False
      ))

  # --- Additional Traces ---
  if additional_traces:
    for trace in additional_traces:
      fig.add_trace(trace)


  # --- Update Layout and Style Plot ---
  fig.update_layout(
      margin=dict(l=0,r=0,b=0,t=0),
      title_automargin=True,
      scene=dict(
          xaxis_title='X [mm]',
          yaxis_title='Y [mm]',
          zaxis_title='Z [mm]',
          xaxis=dict(range=[-200, 200]),
          yaxis=dict(range=[-200, 200]),
          zaxis=dict(range=[-400, 0])
      ),
      title=f'{name if name else ""}',
      title_font_size=15,
      title_x=0.5,
      title_y=0.95,
      title_font_family='Fira Code',
      annotations=[
          dict(
              text=f"Configuration ({theta1:.3f}, {theta2:.3f}, {theta3:.3f}) [rad]",
              showarrow=False,
              xref="paper", yref="paper",
              x=0.5, y=0.93,  # Position below the title
              font=dict(size=15, family="Fira Code")
          )
      ],
      width=800,
      height=600
  )

  fig.show()
  if save_to_html:
    new_name = str.lower(name.replace(' ', '_')) if name else 'delta_robot'
    fig.write_html(f'{new_name}.html', full_html=True, include_plotlyjs='cdn', config={"responsive" : True})


# "UP" position
plot_robot(
    theta1=np.deg2rad(15),
    theta2=np.deg2rad(15),
    theta3=np.deg2rad(15),
    name='UP Position'
)

# "DOWN" position
plot_robot(
  theta1=np.pi/2,
  theta2=np.pi/2,
  theta3=np.pi/2,
  name='DOWN Position'
)

# Another configuration with different angles for each leg
plot_robot(
    theta1=np.pi/4 + 0.1,
    theta2=np.pi/4,
    theta3=np.pi/4 - 0.1,
    name='Different Angles'
)

# Animating Robot Trajectories

In [ ]:
def generate_robot_traces(theta1, theta2, theta3):
  # Calculate the knee joint positions
  knee_1, knee_2, knee_3 = knee_joints(theta1, theta2, theta3)
  # Calculate the platform position
  try:
    platform_position = FK(theta1, theta2, theta3)
  except Exception as e:
    print(f"Exception for FK({theta1}, {theta2}, {theta3}): {e}")
    return
  # Get the base vertices
  bv1, bv2, bv3 = base_vertices()
  # Get the motor positions
  m1, m2, m3 = motor_positions()
  # Get the platform vertices
  pv1, pv2, pv3 = platform_vertices(platform_position)

  base_triangle_trace = go.Mesh3d(
      x=[bv1[0], bv2[0], bv3[0]],
      y=[bv1[1], bv2[1], bv3[1]],
      z=[bv1[2], bv2[2], bv3[2]],
      i=[0], j=[1], k=[2],
      color='lightblue', opacity=0.6,
      name='Base'
  )

  platform_triangle_trace = go.Mesh3d(
      x=[pv1[0], pv2[0], pv3[0]],
      y=[pv1[1], pv2[1], pv3[1]],
      z=[pv1[2], pv2[2], pv3[2]],
      i=[0], j=[1], k=[2],
      color='lightgreen', opacity=1.0,
      name='Platform'
  )

  platform_center_trace = go.Scatter3d(
      x=[platform_position[0]], y=[
          platform_position[1]], z=[platform_position[2]],
      mode='markers', marker=dict(color='red', size=5),
      name='Platform Center'
  )

  knee_traces = []
  for k in [knee_1, knee_2, knee_3]:
      knee_traces.append(go.Scatter3d(
          x=[k[0]], y=[k[1]], z=[k[2]],
          mode='markers',
          marker=dict(color='black', size=5),
          name='Knee Joint' + f'{[knee_1, knee_2, knee_3].index(k) + 1}',
          uid=f'KJ{[knee_1, knee_2, knee_3].index(k) + 1}',
          showlegend=False
      ))

  active_link_traces = []
  knees = [knee_1, knee_2, knee_3]
  for m, k in zip([m1, m2, m3], knees):
      active_link_traces.append(go.Scatter3d(
          x=[m[0], k[0]],
          y=[m[1], k[1]],
          z=[m[2], k[2]],
          mode='lines',
          line=dict(color='purple', width=H/6),
          name='Active Link' + f'{[m1, m2, m3].index(m) + 1}',
          uid=f'AL{[m1, m2, m3].index(m) + 1}',
          showlegend=False
      ))

  passive_link_traces = []
  for k, p in zip(knees, [pv1, pv2, pv3]):
      passive_link_traces.append(go.Scatter3d(
          x=[k[0], p[0]],
          y=[k[1], p[1]],
          z=[k[2], p[2]],
          mode='lines',
          line=dict(color='orange', width=H/2),
          name='Passive Link' + f'{knees.index(k) + 1}',
          uid=f'PL{knees.index(k) + 1}',
          showlegend=False
      ))

  return [base_triangle_trace, platform_triangle_trace, platform_center_trace] + knee_traces + active_link_traces + passive_link_traces

def animate_robot(trajectory: tuple[list[float], list[float], list[float]], additional_traces=None):
    # trajectory is a tuple of (x,y,z) points
    x_t, y_t, z_t = trajectory
    assert len(x_t) == len(y_t) == len(z_t), "Trajectory points must have the same length"
    N = len(x_t)

    # Solve IK for the robot at each point of the trajectory
    theta_trajectory = np.array([IK(x, y, z) for x, y, z in zip(x_t, y_t, z_t)])

    trajectory_trace = go.Scatter3d(
        x=x_t,
        y=y_t,
        z=z_t,
        mode='lines',
        line=dict(color='blue', width=5),
        name='Trajectory'
    )

    # Build animation frames
    frames = []
    for i in range(N):
        # Get joint anles for the current trajectory point using IK
        theta1, theta2, theta3 = theta_trajectory[i]
        # Generate traces for the robot at the current configuration
        robot_traces = generate_robot_traces(theta1, theta2, theta3)
        extra_traces = additional_traces[i] if additional_traces else []
        animation_traces = robot_traces + [trajectory_trace] + extra_traces
        # Create a frame with these traces
        frames.append(go.Frame(data=animation_traces, name=str(i)))

    # Create the initial figure using first trajectory point
    initial_theta1, initial_theta2, initial_theta3 = theta_trajectory[0]
    initial_robot_traces = generate_robot_traces(initial_theta1, initial_theta2, initial_theta3)
    initial_robot_traces.append(trajectory_trace)
    if additional_traces:
        initial_robot_traces += [trace[0] for trace in additional_traces]
    # Create the animation figure
    animation_speed = 5  # milliseconds per frame
    fig = go.Figure(data=initial_robot_traces, frames=frames)

    # Add animation buttons (Play/Pause) to the layout
    fig.update_layout(
        title="Delta Robot Animation",
        updatemenus=[
            dict(
                type="buttons",
                buttons=[
                    dict(
                        label="Play",
                        method="animate",
                        args=[
                            None,
                            {"frame": {"duration": animation_speed, "redraw": True},
                            "fromcurrent": True}
                        ]
                    ),
                    dict(
                        label="Pause",
                        method="animate",
                        args=[
                            [None],
                            {"frame": {"duration": 0, "redraw": True}, "mode": "immediate"}
                        ]
                    )
                ],
                direction="left",
                pad={"r": 10, "t": 87},
                showactive=False,
                x=0.1,
                xanchor="right",
                y=0,
                yanchor="top"
            )
        ],
        sliders = [{
        "currentvalue": {"prefix": "Frame: "},
        "steps": [{
            "args": [
                [str(i)],
                {"frame": {"duration": N, "redraw": True}, "mode": "immediate"}
            ],
            "label": str(i),
            "method": "animate"
        } for i in range(N)]
        }],
        width=800,
        height=800,
        scene=dict(
            xaxis_title='X [mm]',
            yaxis_title='Y [mm]',
            zaxis_title='Z [mm]',
            xaxis=dict(range=[-200, 200], autorange=False),
            yaxis=dict(range=[-200, 200], autorange=False),
            zaxis=dict(range=[-400, 0], autorange=False)
        ),
        uirevision=True
    )

    fig.show()

## Trajectory Planning

In [ ]:
def plot_positions_and_velocities(ee_trajectory, joint_trajectory, ee_vel=None, joint_vels=None):
  if isinstance(ee_trajectory, tuple):
    ee_trajectory = np.column_stack(ee_trajectory)
  if ee_vel is None:
    ee_vel = np.gradient(ee_trajectory, axis=0)
  if joint_vels is None:
    joint_vels = np.gradient(joint_trajectory, axis=1)
  N = ee_trajectory.shape[0]
  # Plot the end effector position and velocity trajectory on a single plot
  fig = make_subplots(rows=2, cols=2, subplot_titles=('End Effector Position Trajectory',
                      'Joint Angles Trajectory', 'End Effector Velocity Trajectory', 'Joint Velocities Trajectory'))

  # Plot the position trajectory
  fig.add_trace(go.Scatter(x=np.arange(N), y=ee_trajectory[:, 0],
                mode='lines', name='X Position [mm]'), row=1, col=1)
  fig.add_trace(go.Scatter(x=np.arange(N), y=ee_trajectory[:, 1],
                mode='lines', name='Y Position [mm]'), row=1, col=1)
  fig.add_trace(go.Scatter(x=np.arange(N), y=ee_trajectory[:, 2],
                mode='lines', name='Z Position [mm]'), row=1, col=1)

  # Plot the velocity trajectory
  fig.add_trace(go.Scatter(x=np.arange(N), y=ee_vel[:, 0],
                mode='lines', name='X Velocity [mm/s]'), row=2, col=1)
  fig.add_trace(go.Scatter(x=np.arange(N), y=ee_vel[:, 1],
                mode='lines', name='Y Velocity [mm/s]'), row=2, col=1)
  fig.add_trace(go.Scatter(x=np.arange(N), y=ee_vel[:, 2],
                mode='lines', name='Z Velocity [mm/s]'), row=2, col=1)

  # Plot the joint angles trajectory
  fig.add_trace(go.Scatter(x=np.arange(N),
                y=joint_trajectory[:, 0], mode='lines', name='Theta1 [rad]'), row=1, col=2)
  fig.add_trace(go.Scatter(x=np.arange(N),
                y=joint_trajectory[:, 1], mode='lines', name='Theta2 [rad]'), row=1, col=2)
  fig.add_trace(go.Scatter(x=np.arange(N),
                y=joint_trajectory[:, 2], mode='lines', name='Theta3 [rad]'), row=1, col=2)

  # Plot the joint velocities trajectory
  fig.add_trace(go.Scatter(x=np.arange(N),
                y=joint_vels[:, 0], mode='lines', name='Theta1_dot [rad/s]'), row=2, col=2)
  fig.add_trace(go.Scatter(x=np.arange(N),
                y=joint_vels[:, 1], mode='lines', name='Theta2_dot [rad/s]'), row=2, col=2)
  fig.add_trace(go.Scatter(x=np.arange(N),
                y=joint_vels[:, 2], mode='lines', name='Theta3_dot [rad/s]'), row=2, col=2)

  # Set the Y axes labels for each subplot
  fig.update_yaxes(title_text='End Effector Position [mm]', row=1, col=1)
  fig.update_yaxes(title_text='Joint Angles [rad]', row=1, col=2)
  fig.update_yaxes(title_text='End Effector Velocity [mm/s]', row=2, col=1)
  fig.update_yaxes(title_text='Joint Velocities [rad/s]', row=2, col=2)

  # Set the X axes label for all subplots
  for r, c in [(1, 1), (1, 2), (2, 1), (2, 2)]:
    fig.update_xaxes(title_text='Time Steps', row=r, col=c)

  fig.update_layout(
      height=500,
      width=1300,
      title_text="End Effector and Joint Trajectories",
      grid=dict(rows=2, columns=2, pattern='independent'),
      margin=dict(l=50, r=50, t=50, b=50)
  )
  fig.show()

### Vertical Oscillation

In [ ]:
# Create a simple up down trajectory with 4 oscillations between
# Z = -100 and Z = -200
num_samples = 300
z_traj = -150 + 50 * np.sin(4 * np.pi * np.linspace(0, 1, num_samples))
x_traj = np.zeros(num_samples)
y_traj = np.zeros(num_samples)

ee_trajectory = (x_traj, y_traj, z_traj)
thetas = np.array([IK(x, y, z) for x, y, z in zip(x_traj, y_traj, z_traj)])

for t in range(ee_trajectory[0].shape[0]):
  ik_result = IK(x_traj[t], y_traj[t], z_traj[t])
  fk_result = FK(*ik_result)
  assert np.allclose(fk_result, [x_traj[t], y_traj[t], z_traj[t]], atol=1e-6), f'FK(IK({x_traj[t]}, {y_traj[t]}, {z_traj[t]})) = {fk_result} != ({x_traj[t]}, {y_traj[t]}, {z_traj[t]})'

animate_robot(ee_trajectory)
plot_positions_and_velocities(ee_trajectory, thetas)

### XY Circle with Z Oscillation

In [ ]:
# Create the circle trajectory in the XY plane
# while the Z coordinate goes through 2 cycles of a sine wave
circle_center_z = -180
amplitude_z = 25
num_samples = 500
t = np.linspace(0, 2 * np.pi, num_samples)
x_circle = 50 * np.cos(t)
y_circle = 50 * np.sin(t)
z_circle = circle_center_z + amplitude_z * np.sin(2 * t)

thetas = np.array([IK(x, y, z) for x, y, z in zip(x_circle, y_circle, z_circle)])
# plot_robot(theta1, theta2, theta3)

ee_trajectory = (x_circle, y_circle, z_circle)
animate_robot(ee_trajectory)
plot_positions_and_velocities(ee_trajectory, thetas)

### FK Trajectory

In [ ]:
# Create a FK trajectory

# theta1(t) = theta_max * sin(t)
# theta2(t) = theta_max * sin(2t)
# theta3(t) = theta_max * sin(3t)
# where theta_max = 45 degrees = pi/4 radians

theta_min = np.deg2rad(15) # Breaks at 8 degrees!!
theta_max = np.deg2rad(85) # Breaks at 90 degrees!!
t = np.linspace(0, 2 * np.pi, 100)
theta1_traj = theta_min + (theta_max - theta_min) * (1 + np.sin(4*t)) / 2
theta2_traj = theta_min + (theta_max - theta_min) * (1 + np.sin(2 * t)) / 2
theta3_traj = theta_min + (theta_max - theta_min) * (1 + np.sin(3 * t)) / 2

print(f'Theta1 trajectory min, max, median: {np.min(theta1_traj):.2f}, {np.max(theta1_traj):.2f}  {np.median(theta1_traj):.2f} [rad]')
print(f'Theta2 trajectory min, max, median: {np.min(theta2_traj):.2f}, {np.max(theta2_traj):.2f} {np.median(theta2_traj):.2f} [rad]')
print(f'Theta3 trajectory min, max, median: {np.min(theta3_traj):.2f}, {np.max(theta3_traj):.2f} {np.median(theta3_traj):.2f} [rad]')

# Compute the FK trajectory
x_traj, y_traj, z_traj = [], [], []
for theta1, theta2, theta3 in zip(theta1_traj, theta2_traj, theta3_traj):
  x, y, z = FK(theta1, theta2, theta3)
  x_traj.append(x)
  y_traj.append(y)
  z_traj.append(z)

ee_trajectory = (x_traj, y_traj, z_traj)
thetas = np.array([IK(x, y, z) for x, y, z in zip(x_traj, y_traj, z_traj)])
animate_robot(ee_trajectory)
plot_positions_and_velocities(ee_trajectory, thetas)

### Axes Trajectory

In [ ]:
# Axes Trajectory
# Start at (0,0, -180)
num_points = 100
x_traj = []
y_traj = []
z_traj = []

# X Axis Translation from (0, 0, -180) to (80, 0, -180)
x_traj.extend(np.linspace(0, 80, num_points))
y_traj.extend(np.zeros(num_points))
z_traj.extend(np.zeros(num_points) - 180)
# Return to (0, 0, -180)
x_traj.extend(np.linspace(80, 0, num_points))
y_traj.extend(np.zeros(num_points))
z_traj.extend(np.zeros(num_points) - 180)

# Y Axis translation from (0, 0, -180) to (0, 80, -180)
x_traj.extend(np.zeros(num_points))
y_traj.extend(np.linspace(0, 80, num_points))
z_traj.extend(np.zeros(num_points) - 180)
# Return to (0, 0, -180)
x_traj.extend(np.zeros(num_points))
y_traj.extend(np.linspace(80, 0, num_points))
z_traj.extend(np.zeros(num_points) - 180)

# Z Axis translation from (0, 0, -180) to (0, 0, -220)
x_traj.extend(np.zeros(num_points))
y_traj.extend(np.zeros(num_points))
z_traj.extend(np.linspace(-180, -220, num_points))
# Return to (0, 0, -180)
x_traj.extend(np.zeros(num_points))
y_traj.extend(np.zeros(num_points))
z_traj.extend(np.linspace(-220, -180, num_points))

# Plot EE and joint trajectory
ee_trajectory = (x_traj, y_traj, z_traj)
thetas = np.array([IK(x, y, z) for x, y, z in zip(x_traj, y_traj, z_traj)])

animate_robot(ee_trajectory)
plot_positions_and_velocities(np.column_stack(ee_trajectory), thetas)

# Workspace

The delta robot's configuration space is bound by the 3 motor angles $\theta_1$, $\theta_2$, and $\theta_3$. The configuration space is a 3D space, where each point in the space corresponds to a unique position of the robot's end effector. The workspace of the robot can be found by solving the forward kinematics equations for the robot's end effector for all possible joint angles in the configuration space.

In [ ]:
# Motor ranges in Dynamixel steps [2107, 2959]
theta1_limit = [0.17453, 1.48353] # radians (10, 85 degrees)
theta2_limit = [0.17453, 1.48353] # radians (10, 85 degrees)
theta3_limit = [0.17453, 1.48353] # radians (10, 85 degrees)

# Define the motor resolution in radians per step (4096 steps per revolution)
resolution = 2 * np.pi / 4096

# Define the motor limits in Dynamixel steps
motor_limits = [
  (2107, 2959), # Theta1
  (2107, 2959), # Theta2
  (2107, 2959)  # Theta3
]

# Draw the workspace by sampling the joint angles and calling FK
workspace_sample_points = 50
theta1_samples = np.linspace(theta1_limit[0], theta1_limit[1], workspace_sample_points)
theta2_samples = np.linspace(theta2_limit[0], theta2_limit[1], workspace_sample_points)
theta3_samples = np.linspace(theta3_limit[0], theta3_limit[1], workspace_sample_points)

workspace_points = []
for theta1 in theta1_samples:
  for theta2 in theta2_samples:
    for theta3 in theta3_samples:
      xyz = FK(theta1, theta2, theta3)
      if xyz is not None:
        workspace_points.append((xyz[0], xyz[1], xyz[2]))

# Extract the X, Y, Z coordinates from the workspace points
x_points = [p[0] for p in workspace_points]
y_points = [p[1] for p in workspace_points]
z_points = [p[2] for p in workspace_points]

# Create a 3D scatter plot trace of the workspace points
workspace_trace = go.Scatter3d(
  x=x_points, y=y_points, z=z_points,
  mode='markers',
  marker=dict(size=2, color=z_points, colorscale='Viridis', opacity=0.8),
  name='Workspace Points',
  showlegend=False
)

# Plot the robot in a specific configuration along with the workspace
plot_robot(
  theta1=np.deg2rad(15),
  theta2=np.deg2rad(15),
  theta3=np.deg2rad(15),
  additional_traces=[workspace_trace],
  save_to_html=False,
  name='Robot Workspace'
)

In [ ]:
from scipy.spatial import ConvexHull

# Get all the workspace points that intersect with a Z plane at Z=-180 and plot them
# Filter points that intersect with the Z=-180 plane (with some tolerance)
z_plane = -180
tolerance = 1.0

plane_points = [p for p in workspace_points if np.isclose(p[2], z_plane, atol=tolerance)]
x_plane_points = [p[0] for p in plane_points]
y_plane_points = [p[1] for p in plane_points]
z_plane_points = [p[2] for p in plane_points]

# Compute the convex hull to determine the actual boundary of the reachable workspace
hull = ConvexHull(np.column_stack((x_plane_points, y_plane_points)))

# Find the bounding box
min_x, max_x = np.min(x_plane_points), np.max(x_plane_points)
min_y, max_y = np.min(y_plane_points), np.max(y_plane_points)

# Define scanning parameters
y_step = 5  # Step size in Y direction
num_points = 100  # Points per scan line

x_traj = []
y_traj = []
z_traj = []

# Generate evenly spaced Y values within the valid range
scan_y_values = np.arange(max_y, min_y, -y_step)

moving_left = True  # Direction flag

# Generate the scan trajectory within the workspace boundary
for y in scan_y_values:
    # Get X values at this Y level
    mask = np.isclose(y_plane_points, y, atol=0.5)
    x_vals_at_y = np.array(x_plane_points)[mask]

    if len(x_vals_at_y) == 0:
        continue  # Skip if no points at this level

    leftmost_x = np.min(x_vals_at_y)
    rightmost_x = np.max(x_vals_at_y)

    # Generate scan path at this Y level
    if moving_left:
        x_traj.extend(np.linspace(rightmost_x, leftmost_x, num_points))
    else:
        x_traj.extend(np.linspace(leftmost_x, rightmost_x, num_points))

    y_traj.extend([y] * num_points)
    z_traj.extend([z_plane] * num_points)

    # Toggle direction for next row
    moving_left = not moving_left

print(f'Number of workspace points at Z=-180: {len(x_plane_points)}')
z_plane_cross_section_trace = go.Scatter3d(
  x=x_plane_points, y=y_plane_points, z=[-180]*len(z_plane_points),
  mode='markers',
  marker=dict(size=2, color='red', opacity=0.6),
  name='Z Plane Cross Section',
  showlegend=False
)

scan_trace = go.Scatter3d(
    x=x_traj, y=y_traj, z=z_traj,
  mode='lines',
  line=dict(color='blue', width=5),
  name='Scan Trajectory',
  showlegend=False
)

t1, t2, t3 = IK(0, 0, -180)
plot_robot(
  theta1=t1,
  theta2=t2,
  theta3=t3,
  additional_traces=[z_plane_cross_section_trace, scan_trace],
  save_to_html=False,
  name='Workspace Cross Section at Z=-180'
)

# Write the scan trajectory to a CSV file
# Scan trajectory should have 3 columns: X, Y, Z
# scan_trajectory = np.column_stack((x_traj, y_traj, z_traj))
# np.savetxt('scan_trajectory.csv', scan_trajectory, delimiter=',', header='X,Y,Z', comments='')
# print(f'Scan trajectory saved to scan_trajectory.csv')

In [ ]:
ee_trajectory = (x_traj, y_traj, z_traj)
thetas = np.array([IK(x, y, z) for x, y, z in zip(x_traj, y_traj, z_traj)])
plot_positions_and_velocities(np.column_stack(ee_trajectory), thetas)

In [ ]:
workspace_convex_hull = ConvexHull(np.array(workspace_points))

# Extract the vertices of the convex hull
hull_vertices = workspace_convex_hull.points[workspace_convex_hull.vertices]

# Create a 3D scatter plot trace for the convex hull vertices
hull_trace = go.Scatter3d(
  x=hull_vertices[:, 0],
  y=hull_vertices[:, 1],
  z=hull_vertices[:, 2],
  mode='markers',
  marker=dict(size=3, color='blue', opacity=0.8),
  name='Workspace Hull Vertices',
  showlegend=False
)

# Create a 3D mesh trace for the convex hull faces
hull_faces = go.Mesh3d(
  x=workspace_convex_hull.points[:, 0],
  y=workspace_convex_hull.points[:, 1],
  z=workspace_convex_hull.points[:, 2],
  i=workspace_convex_hull.simplices[:, 0],
  j=workspace_convex_hull.simplices[:, 1],
  k=workspace_convex_hull.simplices[:, 2],
  color='lightblue',
  opacity=0.5,
  name='Workspace Hull',
  showlegend=False
)

# Plot the robot in a specific configuration along with the workspace hull
plot_robot(
  theta1=np.deg2rad(15),
  theta2=np.deg2rad(15),
  theta3=np.deg2rad(15),
  additional_traces=[hull_trace, hull_faces],
  save_to_html=False,
  name='Workspace Convex Hull'
)

In [ ]:
from scipy.spatial import Delaunay

# Create a Delaunay triangulation from the convex hull points
workspace_delaunay = Delaunay(workspace_convex_hull.points)

def in_workspace(x, y, z):
  # Check if the point (x, y, z) is inside the convex hull using Delaunay triangulation
  point = np.array([x, y, z])
  return workspace_delaunay.find_simplex(point) >= 0

assert in_workspace(0, 0, -180) == True
assert in_workspace(0, 0, -500) == False

# Determine the bounds dynamically using the workspace convex hull
x_range = (np.min(workspace_convex_hull.points[:, 0]), np.max(workspace_convex_hull.points[:, 0]))
y_range = (np.min(workspace_convex_hull.points[:, 1]), np.max(workspace_convex_hull.points[:, 1]))
z_range = (np.min(workspace_convex_hull.points[:, 2]), np.max(workspace_convex_hull.points[:, 2]))
def sample_random_workspace_point():
  # Sample a random point that is inside the convex hull
  while True:
    x = np.random.uniform(x_range[0], x_range[1])
    y = np.random.uniform(y_range[0], y_range[1])
    z = np.random.uniform(z_range[0], z_range[1])
    if in_workspace(x, y, z):
      return x, y, z

In [ ]:
# Randomly sample points within the workspace to build a "random" trajectory
num_random_samples = 100
random_colors = ['red', 'green', 'blue', 'orange', 'purple', 'cyan', 'magenta', 'yellow']
random_points = []
while len(random_points) < num_random_samples:
  random_points.append(sample_random_workspace_point())

# Save the random points to a csv with X, Y, Z columns
random_x, random_y, random_z = zip(*random_points)
random_points_csv = np.column_stack((random_x, random_y, random_z))
np.savetxt('random_points.csv', random_points_csv, delimiter=',', header='X,Y,Z', comments='')

# Create a single Scatter3d trace for all random points
random_points_trace = go.Scatter3d(
  x=[x for x, _, _ in random_points],
  y=[y for _, y, _ in random_points],
  z=[z for _, _, z in random_points],
  mode='markers',
  marker=dict(color=[random_colors[np.random.randint(0, len(random_colors))] for _ in random_points], size=6),
  name='Random Points',
  showlegend=False
)
# for i, p in enumerate(random_points):
#   print(f"Random Point {i+1}: {p}")

# Draw a line connecting the random points in order
random_line_trace = go.Scatter3d(
    x=[x for x, _, _ in random_points],
    y=[y for _, y, _ in random_points],
    z=[z for _, _, z in random_points],
    mode='lines',
    line=dict(color='black', width=2),
    name='Random Line',
    showlegend=False
)

# Call plot_robot once with the combined trace
plot_robot(
  theta1=np.deg2rad(15),
  theta2=np.deg2rad(15),
  theta3=np.deg2rad(15),
  additional_traces=[random_points_trace, random_line_trace],
  save_to_html=False,
  name='Random Points Trajectory'
)

# Delta Robot Jacobian

In [ ]:
class DeltaJacobian:
  def __init__(self):
    self.phi = np.deg2rad([-90, 30, 150])  # Angle from base x axis to motor i [rad]

  def compute_aux_angles(self, theta1, theta2, theta3) -> tuple[list[float], list[float]]:
    # Compute theta_2i and theta_3i
    # return ([theta_21, theta_22, theta_23], [theta_31, theta_32, theta_33])
    px, py, pz = FK(theta1, theta2, theta3)

    # C is a 3x3 vector where column 0 is c_x0, x_y0, c_z0, column 1 is c_x1, c_y1, c_z1, etc.
    columns = []
    for i in range(3):
      R = np.array([
          [np.cos(self.phi[i]),  np.sin(self.phi[i]), 0],
          [-np.sin(self.phi[i]), np.cos(self.phi[i]), 0],
          [0,                   0,                  1]
      ])
      P = np.array([px, py, pz])
      D = np.array([UP - L, 0, 0])
      c_i = R @ P + D
      columns.append(c_i)
    C = np.column_stack(columns)
    # C = [c_x1, c_x2, c_x3]
    #     [c_y1, c_y2, c_y3]
    #     [c_z1, c_z2, c_z3]
    C_x2 = C[0][1]
    C_x3 = C[0][2]
    C_y2 = C[1][1]
    C_y3 = C[1][2]
    C_z2 = C[2][1]
    C_z3 = C[2][2]
    # C_squared = c_xi^2 + c_yi^2 + c_zi^2
    C_squared_2 = C_x2**2 + C_y2**2 + C_z2**2
    C_squared_3 = C_x3**2 + C_y3**2 + C_z3**2
    # theta_3i = arccos(C_yi / ELL)
    t32 = np.arccos(C_y2 / ELL)
    t33 = np.arccos(C_y3 / ELL)
    # k_numerator = c_xi^2 + c_yi^2 + c_zi^2 - L^2 - ELL^2
    # k_denominator = 2 * L * ELL * sin(theta_3i)
    # theta_2i = arccos(k_numerator / k_denominator)
    t22 = np.arccos((C_squared_2 - L**2 - ELL**2) / (2 * L * ELL * np.sin(t32)))
    t23 = np.arccos((C_squared_3 - L**2 - ELL**2) / (2 * L * ELL * np.sin(t33)))
    return ([theta2, t22, t23], [theta3, t32, t33])

  def Jtheta(self, theta1, theta2, theta3):
    (t21, t22, t23), (t31, t32, t33) = self.compute_aux_angles(theta1, theta2, theta3)
    JThetaRight = np.diag([
      np.sin(t21)*np.sin(t31),
      np.sin(t22)*np.sin(t32),
      np.sin(t23)*np.sin(t33)])
    return np.multiply(L, JThetaRight)

  def Jp(self, theta1, theta2, theta3):
    (t21, t22, t23), (t31, t32, t33) = self.compute_aux_angles(theta1, theta2, theta3)
    t1 = np.array([theta1, theta2, theta3])
    t2 = np.array([t21, t22, t23])
    t3 = np.array([t31, t32, t33])
    def J_ix(i):
      return np.sin(t3[i]) * np.cos(t2[i] + t1[i]) * np.cos(self.phi[i]) + np.cos(t3[i]) * np.sin(self.phi[i])
    def J_iy(i):
      return -np.sin(t3[i]) * np.cos(t2[i] + t1[i]) * np.sin(self.phi[i]) + np.cos(t3[i]) * np.cos(self.phi[i])
    def J_iz(i):
      return np.sin(t3[i]) * np.sin(t2[i] + t1[i])
    J = np.zeros((3,3))
    for i in range(3):
      J[i][0] = J_ix(i)
      J[i][1] = J_iy(i)
      J[i][2] = J_iz(i)
    return J

  def theta_dot(self, thetas, end_effector_velocity):
    # Jp * p_dot = Jtheta * theta_dot, so theta_dot = Jtheta_inv * Jp * p_dot
    Jp = self.Jp(*thetas)
    Jtheta = self.Jtheta(*thetas)
    J = np.linalg.inv(Jtheta) @ Jp
    return J @ end_effector_velocity

DJ = DeltaJacobian()
thetas = np.array([np.pi/4, np.pi/4, np.pi/4])
aux = DJ.compute_aux_angles(*thetas)
print(f'aux = {np.round(aux,3)} [rad]')
Jac = DJ.Jp(*thetas)
print(f'Jp =\n{np.round(Jac,3)}')
Jac_theta = DJ.Jtheta(*thetas)
print(f'Jtheta =\n{np.round(Jac_theta,3)}')
p_dot = np.array([1, 0, 0])
theta_dot = DJ.theta_dot(thetas, p_dot)
print(f'theta_dot = {np.round(theta_dot,3)} [rad/s] for p_dot = {p_dot} and thetas = {np.round(thetas,2)} [rad]')

## Converting Position Trajectories to Velocity Trajectories

In [ ]:
def create_joint_velocity_traj(x_traj, y_traj, z_traj, num_sample_points=200):
  t = np.linspace(0, 1, num_sample_points)
  dt = t[1] - t[0]
  # Compute the end effector velocity trajectory
  x_vel = np.gradient(x_traj, dt)
  y_vel = np.gradient(y_traj, dt)
  z_vel = np.gradient(z_traj, dt)
  end_effector_velocity = np.column_stack((x_vel, y_vel, z_vel))

  # Compute the joint angles for each point in the trajectory
  thetas = np.array([IK(x, y, z) for x, y, z in zip(x_traj, y_traj, z_traj)])

  joint_velocities = np.zeros((len(t), 3))
  for i in range(len(t)):
    joint_velocities[i] = DJ.theta_dot(thetas[i], end_effector_velocity[i])

  return joint_velocities

In [ ]:
## Create a position-velocity trajectory

# Get the straight up and down position trajectory
# Create a simple up down trajectory with 4 oscillations between
# Z = -100 and Z = -200
num_samples = 300
center = -150
amplitude = 50
t = np.linspace(0, 1, num_samples)
x_traj = np.zeros(num_samples)
y_traj = np.zeros(num_samples)

# Use a cosine for the position trajectory so that velocity (its derivative) is a sine wave
z_traj = center + amplitude * np.cos(4 * np.pi * t)
# The derivative of cos is -sin. This gives a velocity that starts at zero:
z_vel = -4 * np.pi * amplitude * np.sin(4 * np.pi * t)
# Enforce endpoints if needed (though sin(0)=0 and sin(4*pi)=0 naturally):
z_vel[0] = 0
z_vel[-1] = 0

end_effector_velocity = np.zeros((num_samples, 3))
end_effector_velocity[:, 2] = z_vel

# Compute the joint angles for each point in the trajectory
thetas = np.array([IK(x, y, z) for x, y, z in zip(x_traj, y_traj, z_traj)])

joint_velocities = np.zeros((num_samples, 3))
for i in range(num_samples):
  joint_velocities[i] = DJ.theta_dot(thetas[i], end_effector_velocity[i])

# Print the position and velocity trajectories
print(f"First 5 z positions: {z_traj[:5]} [mm]")
print(f"First 5 z velocities: {z_vel[:5]} [mm/s]")
print(f"First 5 joint angles: {thetas[:5]} [rad]")
print(f"First 5 joint velocities: {joint_velocities[:5]} [rad/s]")

ee_trajectory = np.column_stack((x_traj, y_traj, z_traj))
plot_positions_and_velocities(ee_trajectory, thetas, end_effector_velocity, joint_velocities)

In [ ]:
# Create the circle trajectory in the XY plane
# while the Z coordinate goes through 2 cycles of a sine wave
circle_center_z = -180
amplitude_z = 25
num_samples = 500
t = np.linspace(0, 2 * np.pi, num_samples)
x_circle = 50 * np.cos(t)
y_circle = 50 * np.sin(t)
z_circle = circle_center_z + amplitude_z * np.sin(2 * t)

# Compute the end effector velocity trajectory
x_vel = -50 * np.sin(t)
y_vel = 50 * np.cos(t)
z_vel = 2 * amplitude_z * np.cos(2 * t)
dt = t[1] - t[0]
x_vel = np.gradient(x_circle, dt)
y_vel = np.gradient(y_circle, dt)
z_vel = np.gradient(z_circle, dt)

end_effector_velocity = np.column_stack((x_vel, y_vel, z_vel))

thetas = np.array([IK(x, y, z) for x, y, z in zip(x_circle, y_circle, z_circle)])

joint_velocities = np.zeros((num_samples, 3))
for i in range(num_samples):
  joint_velocities[i] = DJ.theta_dot(thetas[i], end_effector_velocity[i])

ee_trajectory = np.column_stack((x_circle, y_circle, z_circle))
plot_positions_and_velocities(ee_trajectory, thetas, end_effector_velocity, joint_velocities)